In [21]:
import numpy as np
import pandas as pd
import sklearn.datasets as generate_dataset
from sklearn.model_selection import train_test_split

In [22]:
wine = generate_dataset.load_wine()
X = wine['data']
y = wine['target']

X_df = pd.DataFrame(X, columns=wine.feature_names)
y_df = pd.DataFrame(y, columns=["target"])

print(X_df.head())
print(y_df.head())
print(np.unique(y))

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  
0                  

In [23]:
# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
#Calculate Prior Probability
def prior_prob(y, num):
    m = y.shape[0]
    no_of_values = np.sum(y == num)
    return no_of_values / m

priors = {
    0: prior_prob(y_train, 0),
    1: prior_prob(y_train, 1),
    2: prior_prob(y_train, 2)
}
print(priors)
likelihoods = likelihood_params(X_train, y_train)

{0: np.float64(0.31690140845070425), 1: np.float64(0.4014084507042254), 2: np.float64(0.28169014084507044)}


In [25]:
#Calculate Likelihood parameters
def likelihood_params(X, y):
    likelihoods = {}
    for c in np.unique(y):
        X_c = X[y == c]
        means = X_c.mean(axis=0)
        vars = X_c.var(axis=0)
        likelihoods[c] = (means, vars)
    return likelihoods

likelihoods = likelihood_params(X, y)

In [26]:
#Gaussian Probability Density Function
def gaussian_pdf(x, mean, var):
    eps = 1e-6  # small value to prevent division by zero
    coeff = 1.0 / np.sqrt(2.0 * np.pi * (var + eps))
    exponent = np.exp(- (x - mean) ** 2 / (2 * (var + eps)))
    return coeff * exponent

In [27]:
# Prediction Function
def predict(X, priors, likelihoods):
    predictions = []
    for x in X:
        posteriors = {}
        for c, prior in priors.items():
            mean, var = likelihoods[c]
            likelihood = np.prod(gaussian_pdf(x, mean, var))
            posteriors[c] = prior * likelihood
        predictions.append(max(posteriors, key=posteriors.get))
    return np.array(predictions)

In [28]:
# Step 5: Evaluate on Training Set
y_pred = predict(X_test, priors, likelihoods)
accuracy = np.mean(y_pred == y_test)

print("Predicted:", y_pred[:10])
print("Actual:   ", y_test[:10])
print("Test Accuracy:", accuracy)

Predicted: [0 0 2 0 1 0 1 2 1 2]
Actual:    [0 0 2 0 1 0 1 2 1 2]
Test Accuracy: 1.0
